# Notebook 1: The Data Pipeline — Ingestion, Storage, and Cleaning

This notebook explains **how raw data flows through the first stage of the pipeline** and the reasoning behind every design decision. Each section ends with an **Interview Talking Point** — a concise way to explain that decision out loud.

---

## The Big Picture

```
3 raw CSV files (638MB)
        │
        ▼
   [ ingest.py ]
   - Load CSVs
   - Drop nulls & duplicates
   - Write to SQLite
        │
        ▼
  articles.db  ◄── all downstream steps read from here
```

**Why not just keep using the CSVs?**  
Every pipeline step needs the article data. If each step re-reads 638MB of CSVs, that's slow and error-prone. SQLite gives us a single queryable file — think of it as a lightweight database that lives in one `.db` file with no server needed.

In [ ]:
import sqlite3
import time
import pandas as pd

DB_PATH = '../data/processed/articles.db'
conn = sqlite3.connect(DB_PATH)

# Quick check — how many articles made it through ingest?
count = pd.read_sql('SELECT COUNT(*) as total FROM articles', conn).iloc[0, 0]
print(f'Total articles in database: {count:,}')

---

## ADR-001: Why SQLite?

### The problem SQLite solves

We have 638MB of CSV data that every pipeline step needs. The naive approach is to re-read the CSVs every time. Let's see how SQLite compares:

In [ ]:
# Option A: Re-read the CSV each time
t0 = time.time()
df_csv = pd.read_csv('../data/raw/articles1.csv', usecols=['id','title','publication','content'])
csv_time = time.time() - t0
print(f'Reading ONE CSV:      {csv_time:.2f}s  (and we have 3 CSVs)')

# Option B: Query SQLite — only fetch the columns we need
t0 = time.time()
df_sql = pd.read_sql('SELECT id, title, publication, content FROM articles LIMIT 50000', conn)
sql_time = time.time() - t0
print(f'SQLite query (50k):   {sql_time:.2f}s')

print(f'\nSQLite is ~{csv_time/sql_time:.1f}x faster for selective column reads')

In [ ]:
# SQLite also lets us run real SQL queries — very useful for EDA and feature engineering
# Example: get article counts and average content length per publication in one query
query = """
    SELECT
        publication,
        COUNT(*) AS article_count,
        ROUND(AVG(LENGTH(content)), 0) AS avg_content_chars
    FROM articles
    GROUP BY publication
    ORDER BY article_count DESC
"""
pd.read_sql(query, conn)

### Why not PostgreSQL?

PostgreSQL is production-grade but requires a running server, configuration, and connection management. For a local single-machine pipeline, SQLite is the right tool — zero setup, one file, works everywhere.

> **Interview Talking Point (ADR-001):**  
> *"I stored the cleaned articles in SQLite rather than re-reading CSVs in each step. This gave me a persistent, queryable store that every downstream step — transform, train, evaluate — can read from using SQL, without loading 638MB into memory each time."*

---

## ADR-002: Conservative Cleaning Strategy

### What does 'cleaning' mean here?

Raw data is messy. We need to decide: how aggressively do we clean before storing?

**Our rule:** At ingest time, only remove rows that are *definitely* unusable:
1. Missing `content` — there's nothing to model
2. Missing `title` — used as a feature
3. Exact duplicate content — same article appearing twice

Everything else (e.g. very short articles) is deferred to `transform.py`.

In [ ]:
# Let's look at what was removed and why
cleaning_log = {
    'Stage': ['Raw (3 CSVs combined)', 'After drop null content', 'After drop null title', 'After drop duplicates'],
    'Rows':  [142570,                  142570,                     142568,                  142036],
    'Removed': ['-', '0', '2', '532']
}
pd.DataFrame(cleaning_log)

In [ ]:
# Why were there duplicates? Let's look at a real example
# (We can't see the deleted rows, but we can illustrate with near-duplicates that survived)
query = """
    SELECT publication, title, LENGTH(content) as content_len
    FROM articles
    WHERE publication IN ('Reuters', 'Washington Post')
    ORDER BY content_len ASC
    LIMIT 5
"""
print('Shortest articles by wire services (often reprinted elsewhere):')
pd.read_sql(query, conn)

### Why defer length filtering to `transform.py`?

**Separation of concerns** — a principle you already know from software engineering.

- `ingest.py` is responsible for: *"load data faithfully, remove only what is structurally broken"*
- `transform.py` is responsible for: *"prepare data for modeling, apply quality thresholds"*

If we later decide 200 chars is too aggressive a cutoff and want 100 instead, we change one line in `transform.py` — not `ingest.py`. The SQLite database stays stable.

> **Interview Talking Point (ADR-002):**  
> *"I separated ingest cleaning from transform cleaning. Ingest only drops structurally broken rows — nulls and exact duplicates. Length thresholds and NLP-specific filtering live in the transform step. This keeps each module focused on a single responsibility and makes thresholds easy to tune without re-running ingestion."*

---

## ADR-003: Handling Missing Author and Date

About 11% of articles are missing an `author`, and 2% are missing a `date`. What do we do?

In [ ]:
df = pd.read_sql('SELECT * FROM articles', conn)

# Show null counts as percentages
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
print('Null percentage per column:')
print(null_pct[null_pct > 0].to_string())

In [ ]:
# Which publications are responsible for most missing authors?
missing_author = df[df['author'].isnull()].groupby('publication').size().sort_values(ascending=False)
print('Missing author by publication:')
print(missing_author.to_string())

### The three options and why we kept nulls

| Option | Effect on `author` nulls |
|--------|-------------------------|
| Drop rows | Lose 11% of articles — mostly from Reuters, which publishes wire stories without bylines |
| Impute "Unknown" | Fake data — "Unknown" becomes a spurious author name |
| Keep as null | Honest — downstream code handles nulls explicitly |

The key insight: **`author` and `date` are metadata, not features.** The classifier only uses `content` and `title`. Missing metadata doesn't make an article unmodelable.

> **Interview Talking Point (ADR-003):**  
> *"About 11% of articles had no author — mostly Reuters wire stories. Since author and date aren't used as model features, dropping those rows would lose data for no modeling benefit. I kept the nulls and handle them explicitly in EDA and evaluation queries."*

---

## What Gets Passed to the Next Stage?

The output of `ingest.py` is a clean SQLite table. Here's exactly what `transform.py` reads:

In [ ]:
print('Schema of the articles table:')
schema = pd.read_sql("PRAGMA table_info(articles)", conn)
print(schema[['name', 'type']].to_string(index=False))

print('\nSample row:')
sample = pd.read_sql('SELECT id, title, publication, LENGTH(content) as content_chars FROM articles LIMIT 1', conn)
print(sample.to_string(index=False))

In [ ]:
conn.close()

---

## Summary

| Decision | Why |
|----------|-----|
| SQLite over CSV re-reads | Persistent, queryable, fast — no server needed |
| Conservative cleaning at ingest | Separation of concerns; thresholds stay flexible |
| Keep null author/date | Metadata fields not used as model features; no data lost |